In [1]:
import os
os.chdir('../')

In [2]:
import os, torch, json, csv
from pathlib import Path
from PIL import Image
from typing import Optional, List
from torch import nn
from utils.clean_fid import CleanFIDInception
from tqdm import tqdm

def extract_feats_minimal(
    in_root: str,
    out_dir: str,
    model: Optional[nn.Module] = None,
    batch_size: int = 128,
    shard_size: int = 20000,
):
    if model is None:
        model = CleanFIDInception()
    model.eval()

    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    img_paths = sorted([p for p in Path(in_root).rglob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]])

    writer = csv.writer((out/"index.csv").open("w",newline=""))
    writer.writerow(["image_path","shard_id","offset"])

    feats_buf, paths_buf, shard_id, n_ok = [], [], 0, 0
    def flush():
        nonlocal shard_id, feats_buf, paths_buf
        if not feats_buf: return
        feats = torch.cat(feats_buf).cpu()
        torch.save({"feats":feats}, out/f"feats_{shard_id:06d}.pt")
        for i,p in enumerate(paths_buf): writer.writerow([str(p), shard_id, i])
        shard_id+=1; feats_buf.clear(); paths_buf.clear()

    for i in tqdm(range(0,len(img_paths),batch_size), desc="Extract"):
        batch, ok_paths = [], []
        for p in img_paths[i:i+batch_size]:
            try: batch.append(Image.open(p).convert("RGB")); ok_paths.append(p)
            except: pass
        if not batch: continue
        with torch.no_grad(): feats = model.forward(batch)
        feats_buf.append(feats); paths_buf.extend(ok_paths); n_ok+=len(ok_paths)
        if sum(f.shape[0] for f in feats_buf) >= shard_size: flush()
    flush()

    json.dump({"num_images":n_ok,"num_shards":shard_id}, (out/"meta.json").open("w"))
    return n_ok, shard_id


In [ ]:
model = CleanFIDInception()
extract_feats_minimal(
    "/home/scpark/data/imagenet/train",
    "/home/scpark/data/imagenet_feats/train_clean",
    model=model
)


Extract:   0%|          | 0/10010 [00:00<?, ?it/s]